In [2]:
import torch
import torch.nn as nn 


In [14]:
class Model(nn.Module): 
    def __init__(self, n_input_features):
        super(Model, self).__init__()
        self.linear = nn.Linear(n_input_features, 1)
        
    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred
    
model = Model(n_input_features=6)

learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# train the model...

In [8]:
# lazy method (not recommended)
PATH = "models/model.pth"
torch.save(model, PATH)

# load the model 
model = torch.load(PATH) 
model.eval()

for param in model.parameters(): 
    print(param)

Parameter containing:
tensor([[ 0.2623,  0.2673, -0.3308,  0.0523,  0.3555, -0.1813]],
       requires_grad=True)
Parameter containing:
tensor([-0.1037], requires_grad=True)


In [17]:
# recommended method
# save parameters
torch.save(model.state_dict(), 'models/model_dict.pth')

loaded_model = Model(n_input_features=6)
loaded_model.load_state_dict(torch.load('models/model_dict.pth'))
loaded_model.eval()

for param in loaded_model.parameters(): 
    # 6x weigts, 1x bias
    print(param)

Parameter containing:
tensor([[-0.1999, -0.1052, -0.0978, -0.4055,  0.3010, -0.3899]],
       requires_grad=True)
Parameter containing:
tensor([-0.4049], requires_grad=True)


In [18]:
print(loaded_model.state_dict())

OrderedDict([('linear.weight', tensor([[-0.1999, -0.1052, -0.0978, -0.4055,  0.3010, -0.3899]])), ('linear.bias', tensor([-0.4049]))])


In [20]:
# save checkpoint during training 
checkpoint = { 
    "epoch": 90, 
    "model_state": model.state_dict(), 
    "optim_state": optimizer.state_dict(),
}

torch.save(checkpoint, "models/checkpoint.pth")

# load checkpoint
loaded_checkpoint = torch.load("models/checkpoint.pth")

# dict, access by key
epoch = loaded_checkpoint["epoch"]

model = Model(n_input_features=6)
optimizer = torch.optim.SGD(model.parameters(), lr=0)

model.load_state_dict(loaded_checkpoint["model_state"])
optimizer.load_state_dict(loaded_checkpoint["optim_state"])

optimizer.state_dict()

{'state': {},
 'param_groups': [{'lr': 0,
   'momentum': 0,
   'dampening': 0,
   'weight_decay': 0,
   'nesterov': False,
   'maximize': False,
   'foreach': None,
   'differentiable': False,
   'params': [0, 1]}]}

In [21]:
# save on gpu, load on cpu 
device = torch.device('mps')
model.to(device) 
torch.save(model.state_dict(), 'models/model.pth')

device = torch.device('cpu')
model = Model(n_input_features=6)
model.load_state_dict(torch.load('models/model.pth', map_location=device))


<All keys matched successfully>

In [ ]:
# save on cpu, load on gpu
torch.save(model.state_dict(), 'models/model.pth')

device = torch.device('mps')
model = Model(n_input_features=6)
model.load_state_dict(torch.load('models/model.pth', map_location="mps"))
model.to(device)